# 4 — Statistics and exposure

**Theme:** turning a marked record into numbers — task statistics, and the
metrics used for occupational exposure assessment.

Everything here builds on the activities from
[3 — Activities](03-activities.ipynb).

In [ ]:
import aerosoltools as at

elpi = at.load_elpi_file("../../tests/data/Sample_ELPI.txt")
elpi.mark_activities({
    "Background": [("2023-09-07 09:06:50", "2023-09-07 09:07:50")],
    "Emission":   [("2023-09-07 09:07:55", "2023-09-07 09:08:30")],
    "Decay":      [("2023-09-07 09:09:00", "2023-09-07 09:10:50")],
})
elpi.activities

## What can this dataset be summarised on?

Which quantities are available depends on the instrument. A size-resolved
instrument can produce mass and number fractions at any cut diameter; a
single-channel one cannot. `available_metrics` reports what this particular
dataset supports.

In [ ]:
for metric in elpi.available_metrics():
    print(f"{metric.key:12s} {metric.label:28s} [{metric.unit}]")

## One channel at a time

`summarize` gives descriptive statistics for a single channel, split by
activity.

In [ ]:
elpi.summarize()

## Every activity at once

`summarize_activities` is the usual starting point: descriptive statistics per
activity, including duration and — for size-resolved data — size metrics such as
mode and geometric mean diameter.

In [ ]:
elpi.summarize_activities()

You can restrict it to particular metrics or statistics rather than taking the
default set.

In [ ]:
elpi.summarize_activities(metrics=["PNC", "PM2.5"], stats=["mean", "median", "max"])

## Exposure assessment

`summarize_exposure` answers the occupational-hygiene question: given this
measurement, what is the exposure, and how does it compare with a limit?

It reports a time-weighted average over `twa_window` (8 h by convention), the
short-term exceedances over `short_window` (15 min by convention), peak counts,
high percentiles, and the time spent above each limit.

In [ ]:
exposure = elpi.summarize_exposure(
    metric="PM4.2",              # respirable dust
    activities=["Emission"],
    background="Background",     # subtract the mean of this activity
    exposure_hours=None,         # None -> use the measured task duration
    short_limit=1.0,             # STEL, in metric units
    long_limit=1.0,              # 8 h OEL, in metric units
    short_window="15min",
    twa_window="8h",
)
exposure

Three arguments carry most of the meaning:

- **`background`** — either a number, or the name of an activity whose average
  is used. Subtracting a measured background separates the process
  contribution from what was already in the room.
- **`exposure_hours`** — how long the worker is assumed to be exposed. `None`
  uses the measured duration of the activity; give a number to extrapolate a
  short measurement to a full shift.
- **`metric`** — which quantity the limit applies to.

Compare a short task assumed to last the whole shift against the same task
taken at its measured length:

In [ ]:
measured = elpi.summarize_exposure(metric="PM4.2", activities=["Emission"],
                                   background="Background", exposure_hours=None)
full_shift = elpi.summarize_exposure(metric="PM4.2", activities=["Emission"],
                                     background="Background", exposure_hours=8.0)

twa_cols = [c for c in measured.columns if "TWA" in c]

print("assumed duration = measured task length")
print(measured[["Segment"] + twa_cols].to_string(index=False))
print()
print("assumed duration = 8 h")
print(full_shift[["Segment"] + twa_cols].to_string(index=False))

## Choosing a metric

For size-resolved data the metric can be any of:

- `"PNC"` — total number concentration
- `"MASS"` — total mass concentration
- `"PM<x>"`, `"PN<x>"`, `"PS<x>"`, `"PV<x>"` — cumulative mass, number, surface
  or volume below cut diameter `<x>` in µm, e.g. `"PM2.5"`
- `"PM<a>-<b>"` — the band between two diameters, e.g. `"PM1-4"`, computed with
  the EN 481 / ISO 7708 penetration curves

Several activities can be summarised in one call, which is the convenient form
when comparing tasks.

In [ ]:
elpi.summarize_exposure(
    metric="PM10",
    activities=["Emission", "Decay"],
    background="Background",
    long_limit=5.0,
    short_limit=10.0,
)

## Single-channel instruments

A 1D instrument has no size distribution, so the metric is its own measurement —
`"PNC"` for a CPC.

In [ ]:
cpc = at.load_cpc_file("../../tests/data/Sample_CPC_AIM.txt")
cpc.mark_activities({
    "Task": [("2023-08-14 11:14:00", "2023-08-14 11:18:00")],
})

cpc.summarize_exposure(
    metric="PNC",
    activities=["Task"],
    exposure_hours=8.0,
    long_limit=1e4,
    short_limit=2e4,
)

## Saving the result

Both summary functions accept `filename` and append to a CSV or Excel file, so
results from several measurements accumulate in one place.

```python
elpi.summarize_activities(filename="campaign_summary.xlsx",
                          sheet_name="2023-09-07")
```

---

**Next:** [5 — Plotting](05-plotting.ipynb).